In [1]:
from typing import List, Tuple
import numpy as np
from numpy import ndarray
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

from mandala.all import *
set_logging_level(level='warning')

In [2]:
storage = Storage(in_memory=True)

@op(storage)
def get_data() -> Tuple[ndarray, ndarray]:
    return make_classification(random_state=0)

log_calls = True
@op(storage)
def train_tree(X, y, max_features=2, random_state=0) -> DecisionTreeClassifier:
    if log_calls: print('Computing train_tree...')
    return DecisionTreeClassifier(random_state=random_state, max_depth=2, 
                                  max_features=max_features).fit(X, y)

In [3]:
with run(storage): 
    X, y = get_data()
    for i in range(5): 
        train_tree(X, y, random_state=i)

Computing train_tree...


Computing train_tree...


Computing train_tree...


Computing train_tree...


Computing train_tree...


In [4]:
@op(storage)
def eval_forest(trees:List[DecisionTreeClassifier], X, y) -> float:
    if log_calls: print('Computing eval_forest...')
    majority_vote = np.array([tree.predict(X) for tree in trees]).mean(axis=0) >= 0.5
    return round(accuracy_score(y_true=y, y_pred=majority_vote), 2)

with run(storage, lazy=True):
    X, y = get_data()
    trees = [train_tree(X, y, random_state=i) for i in range(10)]
    forest_acc = eval_forest(trees, X, y)
    print(f'Random forest accuracy: {forest_acc}')

Computing train_tree...


Computing train_tree...


Computing train_tree...


Computing train_tree...


Computing train_tree...


Computing eval_forest...


Random forest accuracy: AtomRef(0.9, type=AtomType(float))


In [5]:
@superop(storage)
def train_trees(X, y, max_features=2, n_trees=10) -> List[DecisionTreeClassifier]:
    return [train_tree(X, y, max_features=max_features, random_state=i) for i in range(n_trees)]

In [6]:
with run(storage, lazy=True):
    X, y = get_data()
    trees = train_trees(X=X, y=y)
    forest_acc = eval_forest(trees=trees, X=X, y=y)

In [7]:
log_calls = False
with run(storage, lazy=True):
    X, y = get_data()
    for n_trees in (10, 20):
        trees = train_trees(X=X, y=y, n_trees=n_trees)
        forest_acc = eval_forest(trees=trees, X=X, y=y)

In [8]:
with query(storage) as q:
    X, y = get_data()
    n_trees = Query().named('n_trees')
    trees = train_trees(X, y, n_trees=n_trees)
    forest_acc = eval_forest(trees, X, y).named('forest_acc')
    df = q.get_table(n_trees, forest_acc)
df

,n_trees,forest_acc
0,10,0.90
1,20,0.91


In [9]:
@op(storage)
def eval_tree(tree, X, y) -> float:
    return round(accuracy_score(y_true=y, y_pred=tree.predict(X)), 2)

@superop(storage, version='1') # note the version update!
def train_trees(X, y, max_features=2, n_trees=10) -> List[DecisionTreeClassifier]:
    trees = [train_tree(X, y, max_features=max_features, random_state=i) for i in range(n_trees)]
    for tree in trees: eval_tree(tree, X, y) # new logic
    return trees

with run(storage, lazy=True):
    X, y = get_data()
    for n_trees in (10, 20):
        trees = train_trees(X=X, y=y, n_trees=n_trees)
        forest_acc = eval_forest(trees=trees, X=X, y=y)

In [10]:
with query(storage) as q:
    X, y = get_data()
    tree = train_tree(X, y).named('tree')
    tree_acc = eval_tree(tree, X, y).named('tree_acc')
    df_trees = q.get_table(tree, tree_acc)
    print(df_trees.head())
    with q.branch():
        trees = MakeList(containing=tree, at_index=0).named('trees')
        forest_acc = eval_forest(trees, X, y).named('forest_acc')
        df_forest = q.get_table(trees, forest_acc)
        print(df_forest)

                                                tree  tree_acc
0  DecisionTreeClassifier(max_depth=2, max_featur...      0.56
1  DecisionTreeClassifier(max_depth=2, max_featur...      0.74
2  DecisionTreeClassifier(max_depth=2, max_featur...      0.67
3  DecisionTreeClassifier(max_depth=2, max_featur...      0.58
4  DecisionTreeClassifier(max_depth=2, max_featur...      0.54


                                               trees  forest_acc
0  [DecisionTreeClassifier(max_depth=2, max_featu...        0.90
1  [DecisionTreeClassifier(max_depth=2, max_featu...        0.91


In [11]:
@op(storage)
def get_data(n_samples=CompatArg(default=100)) -> Tuple[ndarray, ndarray]:
    return make_classification(n_samples=n_samples, random_state=0)

In [12]:
with run(storage, lazy=True, autocommit=True):
    X, y = get_data()
    with delete():
        trees = train_trees(X=X, y=y, n_trees=20)